# Reconstruct Complete AmericanAir Conversations

This notebook reconstructs complete support conversations from the extracted TWCS data.
Parent and reply relationships define the conversation structure; timestamps are used for deterministic ordering and validation.
The output is one JSON object per conversation in `data/processed/conversations.jsonl`.

## Imports and configuration

In [1]:
import json
from collections import defaultdict
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/data.csv")
OUTPUT_PATH = Path("../data/processed/conversations.jsonl")
BRAND_HANDLE = "AmericanAir"
ID_COLUMNS = [
    "tweet_id",
    "author_id",
    "response_tweet_id",
    "in_response_to_tweet_id",
]

## Load and validate the extracted source

In [2]:
dtype_map = {column: "string" for column in ID_COLUMNS}
df = pd.read_csv(DATA_PATH, dtype=dtype_map)

required_columns = {
    "tweet_id",
    "author_id",
    "inbound",
    "created_at",
    "text",
    "response_tweet_id",
    "in_response_to_tweet_id",
}
missing_columns = required_columns.difference(df.columns)
assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"

print(f"Source rows: {len(df):,}")
print(f"Source columns: {df.columns.tolist()}")

Source rows: 86,429
Source columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


## Normalize IDs consistently with `002_EDA.ipynb`

In [3]:
for column in ID_COLUMNS:
    df[column] = df[column].str.replace(r"\.0$", "", regex=True)

df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
df["inbound"] = df["inbound"].astype("boolean")

duplicate_tweet_ids = int(df["tweet_id"].duplicated().sum())
assert duplicate_tweet_ids == 0, "Tweet IDs must be unique for relationship reconstruction"
print(f"Unique tweets: {df['tweet_id'].nunique():,}")
print(f"Duplicate tweet IDs: {duplicate_tweet_ids}")

Unique tweets: 86,429
Duplicate tweet IDs: 0


## Build tweet and reply lookups

In [4]:
df["role"] = df["inbound"].map({True: "customer", False: "agent"})

tweet_ids = set(df["tweet_id"].dropna())
tweet_lookup = df.set_index("tweet_id").to_dict(orient="index")
parent_of = df.set_index("tweet_id")["in_response_to_tweet_id"].to_dict()

declared_children_of = defaultdict(set)
for parent_id, response_ids in zip(df["tweet_id"], df["response_tweet_id"]):
    if pd.notna(response_ids):
        for child_id in str(response_ids).split(","):
            child_id = child_id.strip()
            if child_id:
                declared_children_of[parent_id].add(child_id)

children_of = defaultdict(list)
for child_id, parent_id in parent_of.items():
    if pd.notna(parent_id) and parent_id in tweet_ids:
        children_of[parent_id].append(child_id)

for parent_id in children_of:
    children_of[parent_id].sort()

## Validate parent and reply relationships

In [5]:
orphan_tweet_ids = {
    tweet_id
    for tweet_id, parent_id in parent_of.items()
    if pd.notna(parent_id) and parent_id not in tweet_ids
}
unknown_declared_child_ids = {
    child_id
    for child_ids in declared_children_of.values()
    for child_id in child_ids
    if child_id not in tweet_ids
}

broken_relationships = set()
for child_id, parent_id in parent_of.items():
    if pd.notna(parent_id) and parent_id in tweet_ids:
        if child_id not in declared_children_of.get(parent_id, set()):
            broken_relationships.add((parent_id, child_id, "parent_missing_child_declaration"))

for parent_id, child_ids in declared_children_of.items():
    for child_id in child_ids:
        if child_id in tweet_ids and parent_of.get(child_id) != parent_id:
            broken_relationships.add((parent_id, child_id, "child_points_to_different_parent"))

print(f"Orphan tweets whose parent is missing: {len(orphan_tweet_ids):,}")
print(f"Unknown IDs declared as replies: {len(unknown_declared_child_ids):,}")
print(f"Broken reply relationships: {len(broken_relationships):,}")

Orphan tweets whose parent is missing: 162
Unknown IDs declared as replies: 9,829
Broken reply relationships: 0


## Reconstruct root-based conversation chains

In [6]:
root_by_tweet = {}
cycle_tweet_ids = set()

for tweet_id in tweet_ids:
    if tweet_id in root_by_tweet:
        continue

    path = []
    position = {}
    current = tweet_id
    while True:
        if current in root_by_tweet:
            root_id = root_by_tweet[current]
            break
        if current in position:
            cycle_nodes = set(path[position[current]:])
            cycle_tweet_ids.update(cycle_nodes)
            root_id = min(cycle_nodes)
            break

        position[current] = len(path)
        path.append(current)
        parent_id = parent_of.get(current)
        if pd.isna(parent_id) or parent_id not in tweet_ids:
            root_id = current
            break
        current = parent_id

    for path_tweet_id in path:
        root_by_tweet[path_tweet_id] = root_id

conversation_members = defaultdict(set)
for tweet_id, root_id in root_by_tweet.items():
    conversation_members[root_id].add(tweet_id)

print(f"Reconstructed conversations: {len(conversation_members):,}")
print(f"Cyclic reply chains: {len(cycle_tweet_ids):,} tweets")

Reconstructed conversations: 26,388
Cyclic reply chains: 0 tweets


## Order each conversation by reply-chain traversal

In [7]:
def sort_key(tweet_id):
    timestamp = tweet_lookup[tweet_id].get("created_at")
    timestamp_value = timestamp.value if pd.notna(timestamp) else pd.Timestamp.max.value
    return timestamp_value, tweet_id


def order_conversation(root_id, member_ids):
    ordered_ids = []
    visited = set()

    def visit(tweet_id):
        if tweet_id in visited or tweet_id not in member_ids:
            return
        visited.add(tweet_id)
        ordered_ids.append(tweet_id)
        child_ids = sorted(children_of.get(tweet_id, []), key=sort_key)
        for child_id in child_ids:
            visit(child_id)

    visit(root_id)
    for tweet_id in sorted(member_ids - visited, key=sort_key):
        visit(tweet_id)
    return ordered_ids

ordered_conversations = {
    root_id: order_conversation(root_id, member_ids)
    for root_id, member_ids in conversation_members.items()
}

## Select genuine AmericanAir support conversations

In [8]:
customer_ids = set(df.loc[df["inbound"] == True, "tweet_id"])
american_air_ids = set(
    df.loc[
        (df["author_id"] == BRAND_HANDLE) & (df["inbound"] == False),
        "tweet_id",
    ]
)
direct_support_edges = {
    (parent_id, child_id)
    for child_id, parent_id in parent_of.items()
    if child_id in american_air_ids and parent_id in customer_ids
}

american_conversation_ids = {
    root_id
    for root_id, member_ids in conversation_members.items()
    if member_ids & american_air_ids
}
both_sides_conversation_ids = {
    root_id
    for root_id, member_ids in conversation_members.items()
    if member_ids & customer_ids and member_ids & american_air_ids
}
support_conversation_ids = {
    root_by_tweet[child_id]
    for _, child_id in direct_support_edges
}

print(f"Conversations containing AmericanAir: {len(american_conversation_ids):,}")
print(f"Conversations containing both sides: {len(both_sides_conversation_ids):,}")
print(f"Genuine support conversations with an AmericanAir reply to a customer: {len(support_conversation_ids):,}")

Conversations containing AmericanAir: 26,388
Conversations containing both sides: 26,388
Genuine support conversations with an AmericanAir reply to a customer: 26,388


## Build complete conversation records

In [9]:
def isoformat_or_none(value):
    return value.isoformat() if pd.notna(value) else None


conversations = []
for conversation_id in sorted(support_conversation_ids):
    ordered_ids = ordered_conversations[conversation_id]
    messages = []
    for tweet_id in ordered_ids:
        row = tweet_lookup[tweet_id]
        messages.append(
            {
                "tweet_id": str(tweet_id),
                "author_id": str(row["author_id"]),
                "role": row["role"],
                "inbound": bool(row["inbound"]),
                "created_at": isoformat_or_none(row["created_at"]),
                "text": row["text"] if pd.notna(row["text"]) else "",
            }
        )

    timestamps = [message["created_at"] for message in messages if message["created_at"]]
    conversations.append(
        {
            "conversation_id": str(conversation_id),
            "brand": BRAND_HANDLE,
            "has_customer": True,
            "has_agent": True,
            "start_time": min(timestamps) if timestamps else None,
            "end_time": max(timestamps) if timestamps else None,
            "messages": messages,
        }
    )

print(f"Output conversation records: {len(conversations):,}")

Output conversation records: 26,388


## Conversation statistics and reconstruction validation

In [10]:
conversation_lengths = pd.Series([len(item["messages"]) for item in conversations], dtype="int64")
output_tweet_ids = [
    message["tweet_id"]
    for conversation in conversations
    for message in conversation["messages"]
]
output_tweet_id_counts = pd.Series(output_tweet_ids).value_counts()
expected_output_ids = set().union(
    *(conversation_members[conversation_id] for conversation_id in support_conversation_ids)
)

chronology_violations = 0
relationship_order_violations = 0
for conversation in conversations:
    message_ids = [message["tweet_id"] for message in conversation["messages"]]
    message_positions = {tweet_id: index for index, tweet_id in enumerate(message_ids)}
    for index, child_id in enumerate(message_ids):
        parent_id = parent_of.get(child_id)
        if pd.notna(parent_id) and parent_id in message_positions:
            if message_positions[parent_id] >= index:
                relationship_order_violations += 1
            parent_time = tweet_lookup[parent_id].get("created_at")
            child_time = tweet_lookup[child_id].get("created_at")
            if pd.notna(parent_time) and pd.notna(child_time) and child_time < parent_time:
                chronology_violations += 1

assert set(output_tweet_ids) == expected_output_ids, "Messages were silently dropped or added"
assert len(output_tweet_ids) == len(set(output_tweet_ids)), "Duplicate tweet IDs in output"
assert not cycle_tweet_ids.intersection(set(output_tweet_ids)), "Cyclic chains cannot be serialized safely"
assert relationship_order_violations == 0, "A child appears before its parent"

statistics = {
    "total_source_rows": len(df),
    "total_unique_tweets": len(tweet_ids),
    "total_reconstructed_conversations": len(conversation_members),
    "conversations_containing_americanair": len(american_conversation_ids),
    "conversations_containing_both_customer_and_americanair": len(both_sides_conversation_ids),
    "output_conversations": len(conversations),
    "minimum_conversation_length": int(conversation_lengths.min()) if len(conversation_lengths) else 0,
    "maximum_conversation_length": int(conversation_lengths.max()) if len(conversation_lengths) else 0,
    "average_conversation_length": float(conversation_lengths.mean()) if len(conversation_lengths) else 0.0,
    "median_conversation_length": float(conversation_lengths.median()) if len(conversation_lengths) else 0.0,
    "single_message_conversations": int((conversation_lengths == 1).sum()),
    "multiple_message_conversations": int((conversation_lengths > 1).sum()),
    "orphan_tweets": len(orphan_tweet_ids),
    "broken_reply_relationships": len(broken_relationships),
    "duplicate_tweet_ids": duplicate_tweet_ids,
    "cyclic_reply_chain_tweets": len(cycle_tweet_ids),
    "duplicate_tweet_ids_in_output": int((output_tweet_id_counts > 1).sum()),
    "chronology_violations_on_parent_edges": chronology_violations,
    "relationship_order_violations": relationship_order_violations,
}

print(pd.Series(statistics))

total_source_rows                                         86429.000000
total_unique_tweets                                       86429.000000
total_reconstructed_conversations                         26388.000000
conversations_containing_americanair                      26388.000000
conversations_containing_both_customer_and_americanair    26388.000000
output_conversations                                      26388.000000
minimum_conversation_length                                   2.000000
maximum_conversation_length                                 204.000000
average_conversation_length                                   3.275315
median_conversation_length                                    2.000000
single_message_conversations                                  0.000000
multiple_message_conversations                            26388.000000
orphan_tweets                                               162.000000
broken_reply_relationships                                    0.000000
duplic

## Inspect real conversations

In [11]:
conversation_by_id = {item["conversation_id"]: item for item in conversations}

def show_conversation(conversation_id):
    conversation = conversation_by_id.get(str(conversation_id))
    if conversation is None:
        raise KeyError(f"Conversation not found: {conversation_id}")

    print("=" * 60)
    print(f"CONVERSATION: {conversation['conversation_id']}")
    for message in conversation["messages"]:
        role = message["role"].upper()
        created_at = message["created_at"] or "unknown time"
        created_at = created_at.replace("+00:00", "")[:16]
        safe_text = str(message["text"]).encode("ascii", "backslashreplace").decode("ascii")
        print(f"\n[{role}] {created_at}")
        print(safe_text)

multi_turn_ids = [
    item["conversation_id"]
    for item in conversations
    if len(item["messages"]) > 1
]
long_conversation_ids = [
    item["conversation_id"]
    for item in conversations
    if len(item["messages"]) >= 4
]

print("Five random multi-turn conversations:")
for conversation_id in pd.Series(multi_turn_ids).sample(min(5, len(multi_turn_ids)), random_state=42):
    show_conversation(conversation_id)

print("Three conversations containing at least four messages:")
for conversation_id in long_conversation_ids[:3]:
    show_conversation(conversation_id)

Five random multi-turn conversations:
CONVERSATION: 1194840

[CUSTOMER] 2017-10-25T01:40
@AmericanAir time after time your flights are delayed because your flight attendants are not on time!#notgoodenough #badservice

[AGENT] 2017-10-25T01:48
@400763 Our team may be delayed on an incoming flight, Ed. Please share your flight number, if we can help with an update.
CONVERSATION: 2134117

[CUSTOMER] 2017-11-26T13:08
@AmericanAir, y\u2019all are assholes. What the fuck kind of airline\u2019s economy tickets don\u2019t include a carry-on?? Gtf \U0001f644

[CUSTOMER] 2017-11-26T13:09
@AmericanAir Talking about one personal item... WHAT

[AGENT] 2017-11-26T13:12
@627889 It sounds like you purchased a Basic Economy fare. For more info see: https://t.co/MKvFFVWJdz

[CUSTOMER] 2017-11-26T13:13
@AmericanAir I know wtf i purchased. https://t.co/PK7PkIZX88
CONVERSATION: 1946148

[CUSTOMER] 2017-10-30T10:39
Rise and fly! Oh, how I've missed you @AmericanAir \U0001f60d\U0001f60d

[AGENT] 2017-10-30T1

## Save one JSON object per line

In [12]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as file:
    for conversation in conversations:
        file.write(json.dumps(conversation, ensure_ascii=False) + "\n")

print(f"Wrote {len(conversations):,} conversations to {OUTPUT_PATH}")

Wrote 26,388 conversations to ..\data\processed\conversations.jsonl


## Reload and validate the JSONL output

In [13]:
with OUTPUT_PATH.open(encoding="utf-8") as file:
    reloaded_conversations = [json.loads(line) for line in file if line.strip()]

assert len(reloaded_conversations) == len(conversations)
assert all(item["brand"] == BRAND_HANDLE for item in reloaded_conversations)
assert all(item["messages"] for item in reloaded_conversations)
assert all(
    set(message) >= {"tweet_id", "author_id", "role", "inbound", "created_at", "text"}
    for item in reloaded_conversations
    for message in item["messages"]
)

reloaded_ids = [
    message["tweet_id"]
    for item in reloaded_conversations
    for message in item["messages"]
]
assert set(reloaded_ids) == set(output_tweet_ids)
assert len(reloaded_ids) == len(set(reloaded_ids))
print(f"Reloaded and validated {len(reloaded_conversations):,} JSONL records")

Reloaded and validated 26,388 JSONL records


## Final summary

In [14]:
print(f"Source tweets: {statistics['total_unique_tweets']:,}")
print(f"AmericanAir conversations: {statistics['output_conversations']:,}")
print(f"Multi-turn conversations: {statistics['multiple_message_conversations']:,}")
print(f"Total messages in output: {len(output_tweet_ids):,}")
print(f"Orphan replies: {statistics['orphan_tweets']:,}")
print(f"Broken relationships: {statistics['broken_reply_relationships']:,}")
print(f"Cycles detected: {statistics['cyclic_reply_chain_tweets']:,}")
print(f"\nOutput:\n{OUTPUT_PATH}")

Source tweets: 86,429
AmericanAir conversations: 26,388
Multi-turn conversations: 26,388
Total messages in output: 86,429
Orphan replies: 162
Broken relationships: 0
Cycles detected: 0

Output:
..\data\processed\conversations.jsonl
